# X-Raydar Chest X-ray Multi-Label Classifier + Grad-CAM CrossCheck
**ResNet152 | 37 consensus findings | 30 epochs | radiologist bounding-box IoU**

This replaces NIH ChestX-ray14 with the X-Raydar Multimodal dataset (979 images, 37 consensus labels, radiologist boxes). The Hugging Face dataset card reports a total size of about 2.81 GB. This notebook downloads the annotation JSONL and only a configurable sample of individual PNGs (default 200), not the full dataset snapshot.

Dataset: https://huggingface.co/datasets/dnamodel/xraydar-multimodal

**License:** marked non-commercial; check the dataset terms and cite its authors. This small sample is an educational pipeline demonstration, not a clinical benchmark or diagnostic tool.

## 1. Install packages

In [ ]:
!pip -q install -U huggingface_hub pandas numpy pillow matplotlib scikit-learn tqdm

## 2. Imports and configuration

In [ ]:
import os, json, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, average_precision_score

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
REPO_ID="https://github.com/Bharath17-10/Chest-X-ray-Multi-Label-Classifier.git"
MAX_IMAGES=200       # Increase if you have time/storage
DOWNLOAD_WORKERS=6
IMAGE_SIZE=224
EPOCHS=30
BATCH_SIZE=8         # set 4 if CUDA out-of-memory
LR=1e-4
THRESHOLD=0.5
ROOT=Path("/content/xraydar_gradcam")
DATA_DIR=ROOT/"data"; OUT_DIR=ROOT/"outputs"
DATA_DIR.mkdir(parents=True,exist_ok=True); OUT_DIR.mkdir(parents=True,exist_ok=True)
print("Device:",DEVICE,"| image sample:",MAX_IMAGES,"| epochs:",EPOCHS)

## 3. Download annotation JSONL (small file)

In [ ]:
ann_path=hf_hub_download(REPO_ID, "annotations.jsonl", repo_type="dataset", local_dir=str(DATA_DIR))
records=[json.loads(line) for line in open(ann_path,encoding="utf-8") if line.strip()]
print("Annotation records:",len(records))
print("Example:",records[0]["image_file"], records[0].get("consensus_labels",[])[:5])

## 4. Select a manageable multi-label sample

In [ ]:
records=[r for r in records if r.get("image_file") and isinstance(r.get("consensus_labels"),list)]
all_classes=sorted({str(y) for r in records for y in r.get("consensus_labels",[])})
# Greedy coverage: take a few examples from each rarer label first, then fill randomly.
chosen=[]; chosen_ids=set()
for label in sorted(all_classes,key=lambda z:sum(z in r.get("consensus_labels",[]) for r in records)):
    candidates=[i for i,r in enumerate(records) if label in r.get("consensus_labels",[]) and i not in chosen_ids]
    random.shuffle(candidates)
    for i in candidates[:2]:
        if len(chosen)>=MAX_IMAGES: break
        chosen.append(records[i]); chosen_ids.add(i)
    if len(chosen)>=MAX_IMAGES: break
rest=[i for i in range(len(records)) if i not in chosen_ids]
random.shuffle(rest)
for i in rest:
    if len(chosen)>=MAX_IMAGES: break
    chosen.append(records[i]); chosen_ids.add(i)
print("Selected records:",len(chosen),"of",len(records))

## 5. Download selected images individually (no full snapshot)

In [ ]:
def get_one(r):
    p=hf_hub_download(REPO_ID, r["image_file"], repo_type="dataset", local_dir=str(DATA_DIR))
    return str(r["xray_id"]),p

paths={}; errors=[]
with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
    fs={pool.submit(get_one,r):r for r in chosen}
    for f in tqdm(as_completed(fs),total=len(fs),desc="Selected X-ray images"):
        try:
            image_id,path=f.result(); paths[image_id]=path
        except Exception as e:
            errors.append((fs[f].get("image_file"),str(e)))
print("Downloaded:",len(paths),"Failed:",len(errors))
if errors: print("First errors:",errors[:5])
if len(paths)<20: raise RuntimeError("Too few images downloaded; check Hugging Face access.")

## 6. Build multi-label table and split

In [ ]:
rows=[]
for r in chosen:
    image_id=str(r["xray_id"])
    if image_id in paths:
        rows.append({"image_id":image_id,"image_path":paths[image_id],
                     "labels":r.get("consensus_labels",[]),
                     "boxes":r.get("bounding_boxes",[])})
df=pd.DataFrame(rows)
CLASS_NAMES=sorted({str(label) for labels in df.labels for label in labels})
N_CLASSES=len(CLASS_NAMES); class_to_idx={c:i for i,c in enumerate(CLASS_NAMES)}
def target(labels):
    y=np.zeros(N_CLASSES,dtype=np.float32)
    for label in labels:
        if label in class_to_idx: y[class_to_idx[label]]=1.
    return y
df["target"]=df.labels.apply(target)
print("Usable images:",len(df),"| Classes in sample:",N_CLASSES)
display(pd.DataFrame({"finding":CLASS_NAMES,"positives":np.stack(df.target).sum(0).astype(int)}))
train_df,temp_df=train_test_split(df,test_size=.30,random_state=SEED)
val_df,test_df=train_test_split(temp_df,test_size=.50,random_state=SEED)
train_df=train_df.reset_index(drop=True); val_df=val_df.reset_index(drop=True); test_df=test_df.reset_index(drop=True)
print("Train/val/test:",len(train_df),len(val_df),len(test_df))

## 7. Dataset and DataLoaders

In [ ]:
train_tf=transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.RandomHorizontalFlip(.5),
    transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
eval_tf=transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
class XrayDataset(Dataset):
    def __init__(self,frame,tf): self.frame=frame.reset_index(drop=True); self.tf=tf
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]
        x=self.tf(Image.open(r.image_path).convert("RGB"))
        return x,torch.tensor(r.target,dtype=torch.float32),r.image_id
train_loader=DataLoader(XrayDataset(train_df,train_tf),batch_size=BATCH_SIZE,shuffle=True,num_workers=2)
val_loader=DataLoader(XrayDataset(val_df,eval_tf),batch_size=BATCH_SIZE,shuffle=False,num_workers=2)
test_loader=DataLoader(XrayDataset(test_df,eval_tf),batch_size=BATCH_SIZE,shuffle=False,num_workers=2)
print("Loaders ready")

## 8. ResNet152 model and loss

In [ ]:
model=models.resnet152(weights=models.ResNet152_Weights.DEFAULT)
model.fc=nn.Linear(model.fc.in_features,N_CLASSES)
model=model.to(DEVICE)
yt=np.stack(train_df.target)
pos=yt.sum(0); neg=len(yt)-pos
pos_weight=torch.tensor(np.clip(neg/np.maximum(pos,1),1,30),dtype=torch.float32,device=DEVICE)
criterion=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="min",factor=.5,patience=2)
scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=="cuda")
print("ResNet152 ready:",N_CLASSES,"labels")

## 9. Training functions

In [ ]:
def epoch_run(loader,training=False):
    model.train(training); total=0.; probs=[]; targets=[]
    for x,y,_ in tqdm(loader,leave=False):
        x=x.to(DEVICE); y=y.to(DEVICE)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.cuda.amp.autocast(enabled=DEVICE.type=="cuda"):
                logits=model(x); loss=criterion(logits,y)
            if training:
                scaler.scale(loss).backward(); scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),5.)
                scaler.step(optimizer); scaler.update()
        total+=loss.item()*len(x)
        probs.append(torch.sigmoid(logits).detach().cpu().numpy()); targets.append(y.detach().cpu().numpy())
    return total/len(loader.dataset),np.concatenate(probs),np.concatenate(targets)

## 10. Train for 30 epochs

In [ ]:
history={"train":[],"val":[]}; best=float("inf"); BEST=OUT_DIR/"resnet152_best.pth"
for ep in range(EPOCHS):
    tl,_,_=epoch_run(train_loader,True)
    vl,_,_=epoch_run(val_loader,False); scheduler.step(vl)
    history["train"].append(tl); history["val"].append(vl)
    print(f"Epoch {ep+1}/{EPOCHS} | train {tl:.4f} | val {vl:.4f}")
    if vl<best:
        best=vl
        torch.save({"model":model.state_dict(),"classes":CLASS_NAMES},BEST)
print("Best checkpoint:",BEST)

## 11. Training curve

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(history["train"],label="Train"); plt.plot(history["val"],label="Validation")
plt.xlabel("Epoch"); plt.ylabel("BCE loss"); plt.title("ResNet152 training"); plt.legend(); plt.grid(); plt.show()

## 12. Multi-label test metrics

In [ ]:
ck=torch.load(BEST,map_location=DEVICE); model.load_state_dict(ck["model"]); model.eval()
test_loss,preds_prob,y_true=epoch_run(test_loader,False)
y_pred=(preds_prob>=THRESHOLD).astype(int)
print("Test loss:",round(test_loss,4))
print("Micro F1:",f1_score(y_true,y_pred,average="micro",zero_division=0))
print("Macro F1:",f1_score(y_true,y_pred,average="macro",zero_division=0))
metric_rows=[]
for i,c in enumerate(CLASS_NAMES):
    auc=average_precision=np.nan
    if len(np.unique(y_true[:,i]))>1:
        auc=roc_auc_score(y_true[:,i],preds_prob[:,i])
        average_precision=average_precision_score(y_true[:,i],preds_prob[:,i])
    metric_rows.append({"Finding":c,"AUC":auc,"Average precision":average_precision,
        "F1":f1_score(y_true[:,i],y_pred[:,i],zero_division=0),
        "Precision":precision_score(y_true[:,i],y_pred[:,i],zero_division=0),
        "Recall":recall_score(y_true[:,i],y_pred[:,i],zero_division=0),
        "Positive support":int(y_true[:,i].sum())})
metrics_df=pd.DataFrame(metric_rows); display(metrics_df)
metrics_df.to_csv(OUT_DIR/"classification_metrics.csv",index=False)

## 13. Grad-CAM implementation

In [ ]:
class GradCAM:
    def __init__(self,model,layer):
        self.model=model; self.a=None; self.g=None
        self.h1=layer.register_forward_hook(self.fwd)
        self.h2=layer.register_full_backward_hook(self.bwd)
    def fwd(self,m,i,o): self.a=o
    def bwd(self,m,gi,go): self.g=go[0]
    def generate(self,x,class_idx):
        self.model.zero_grad(set_to_none=True)
        logits=self.model(x); logits[:,class_idx].sum().backward()
        w=self.g.mean((2,3),keepdim=True)
        cam=F.relu((w*self.a).sum(1,keepdim=True))
        cam=F.interpolate(cam,size=x.shape[-2:],mode="bilinear",align_corners=False)[0,0].detach().cpu().numpy()
        cam=(cam-cam.min())/(cam.max()-cam.min()+1e-8)
        return torch.sigmoid(logits)[0].detach().cpu().numpy(),cam
gradcam=GradCAM(model,model.layer4[-1])

## 14. Ground-truth box masks and IoU

In [ ]:
def box_mask(image,boxes,label,size=IMAGE_SIZE):
    w,h=image.size; mask=np.zeros((size,size),dtype=np.uint8)
    for b in boxes:
        if str(b.get("label",""))!=label: continue
        x1=int(np.clip(round(float(b["x_min"])/w*size),0,size-1))
        y1=int(np.clip(round(float(b["y_min"])/h*size),0,size-1))
        x2=int(np.clip(round(float(b["x_max"])/w*size),0,size))
        y2=int(np.clip(round(float(b["y_max"])/h*size),0,size))
        if x2>x1 and y2>y1: mask[y1:y2,x1:x2]=1
    return mask
def cam_mask(cam,pct=80): return (cam>=np.percentile(cam,pct)).astype(np.uint8)
def iou(a,b):
    u=np.logical_or(a,b).sum()
    return float(np.logical_and(a,b).sum()/u) if u else np.nan

## 15. Grad-CAM / radiologist-box cross-check

In [ ]:
iou_rows=[]
for _,r in tqdm(test_df.iterrows(),total=len(test_df),desc="Grad-CAM cross-check"):
    im=Image.open(r.image_path).convert("RGB")
    x=eval_tf(im).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): p=torch.sigmoid(model(x))[0].cpu().numpy()
    for ci in np.where(p>=THRESHOLD)[0]:
        label=CLASS_NAMES[ci]; gt=box_mask(im,r.boxes,label)
        if gt.sum()==0: continue  # no matching annotation: exclude, not IoU=0
        _,cam=gradcam.generate(x,int(ci)); score=iou(cam_mask(cam),gt)
        iou_rows.append({"image_id":r.image_id,"finding":label,"probability":float(p[ci]),
                         "IoU":score,"IoU trust score (%)":100*score})
iou_df=pd.DataFrame(iou_rows)
print("Matched prediction/box pairs:",len(iou_df))
if len(iou_df):
    display(iou_df.head(20))
    per_class=iou_df.groupby("finding").agg(cases=("IoU","count"),mean_IoU=("IoU","mean"),
        median_IoU=("IoU","median"),mean_trust_percent=("IoU trust score (%)","mean")).reset_index()
    display(per_class)
    iou_df.to_csv(OUT_DIR/"gradcam_iou_per_prediction.csv",index=False)
    per_class.to_csv(OUT_DIR/"gradcam_iou_per_class.csv",index=False)
else:
    print("No positive prediction had a matching box in this test sample.")

## 16. Visualize one matching Grad-CAM and bounding box

In [ ]:
shown=False
for _,r in test_df.iterrows():
    im=Image.open(r.image_path).convert("RGB"); x=eval_tf(im).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): p=torch.sigmoid(model(x))[0].cpu().numpy()
    for ci in sorted(np.where(p>=THRESHOLD)[0],key=lambda j:p[j],reverse=True):
        gt=box_mask(im,r.boxes,CLASS_NAMES[ci])
        if gt.sum()==0: continue
        _,cam=gradcam.generate(x,int(ci)); im224=im.resize((IMAGE_SIZE,IMAGE_SIZE))
        plt.figure(figsize=(14,4))
        plt.subplot(1,3,1); plt.imshow(im224); plt.title("Chest X-ray"); plt.axis("off")
        plt.subplot(1,3,2); plt.imshow(im224); plt.imshow(cam,cmap="jet",alpha=.45); plt.title("Grad-CAM: "+CLASS_NAMES[ci]); plt.axis("off")
        plt.subplot(1,3,3); plt.imshow(im224); plt.imshow(gt,cmap="Reds",alpha=.45); plt.title("Radiologist box"); plt.axis("off")
        plt.show(); shown=True; break
    if shown: break
if not shown: print("No positive predicted class matched a box at this threshold.")

## 17. Save model and results

In [ ]:
FINAL=OUT_DIR/"resnet152_xraydar_gradcam_final.pth"
torch.save({"model":model.state_dict(),"classes":CLASS_NAMES,"threshold":THRESHOLD},FINAL)
print("Saved:",FINAL)
print("Output files:",[p.name for p in OUT_DIR.iterdir()])

## 18. Interpretation and limitations

- Dataset used: **X-Raydar Multimodal**, not NIH ChestX-ray14.
- The full dataset contains 979 images and 37 consensus image-level findings. This notebook downloads only up to 200 selected individual images.
- The model uses sigmoid outputs and binary cross-entropy for multi-label classification.
- A small sample can omit rare labels from the test split; AUC is shown as NaN if a class has only one target value in the test split.
- IoU is computed only for predicted labels with a matching radiologist box. Missing boxes are not assigned zero.
- Grad-CAM thresholding (top 20% heatmap pixels) is heuristic; IoU is an overlap score, not a calibrated clinical trust probability.
- The dataset license is non-commercial. Follow the dataset card's conditions and cite its authors.
- This is a research/education prototype, not a clinical diagnostic tool.